# Решения. Отчёт по таблице объявлений

Эталон итоговой пары модуля. Ученикам до сдачи не показывать. Домашнего задания у этой пары нет.


In [ ]:
from pathlib import Path
import pandas as pd

LISTINGS_URL = (
    "https://raw.githubusercontent.com/gurovic/letovo-ml-profile/main/"
    "modules/08_02_carsharing_pandas_lr/data/listings.csv"
)

def find_listings_csv():
    for p in (
        Path("listings.csv"),
        Path("../listings.csv"),
        Path("../../data/listings.csv"),
        Path("../../../data/listings.csv"),
        Path("../data/listings.csv"),
    ):
        if p.exists():
            return p.resolve()
    return LISTINGS_URL

LISTINGS_PATH = find_listings_csv()
df = pd.read_csv(LISTINGS_PATH)


## 1. Размер таблицы


In [ ]:
n_rows = int(len(df))
n_cols = int(df.shape[1])
REPORT_TABLE = (
    f'В таблице {n_rows} строк и {n_cols} столбцов: каждая строка — объявление аренды, '
    'а цена ночи лежит в столбце price.'
)
assert n_rows == len(df) and n_cols == df.shape[1]
assert str(n_rows) in REPORT_TABLE and str(n_cols) in REPORT_TABLE
print(REPORT_TABLE)


## 2. Пустые ячейки


In [ ]:
missing = df.isna().sum()
gaps = [name for name, n in missing.items() if int(n) > 0]
if gaps:
    REPORT_NA = 'Пустые ячейки есть в столбцах: ' + ', '.join(gaps) + '.'
else:
    REPORT_NA = 'Пустых ячеек нет ни в одном столбце.'
assert len(missing) == df.shape[1]
print(missing)
print(REPORT_NA)


## 3. Цена по всей таблице


In [ ]:
price_mean = float(df['price'].mean())
price_median = float(df['price'].median())
price_min = float(df['price'].min())
price_max = float(df['price'].max())
which = 'среднее выше медианы' if price_mean > price_median else 'медиана не ниже среднего'
REPORT_PRICE = (
    f'Средняя цена {price_mean:.1f}, медиана {price_median:.1f}, '
    f'минимум {price_min:.0f}, максимум {price_max:.0f}. '
    f'На этих данных {which}: дорогие объявления тянут среднее вверх, если среднее больше медианы.'
)
assert abs(price_mean - float(df['price'].mean())) < 0.01
print(REPORT_PRICE)


## 4. Срез: объявления хотя бы на четырёх гостей


In [ ]:
fits = df.loc[df['accommodates'] >= 4]
n_fits_four = int(len(fits))
median_fits_four = float(fits['price'].median())
first_area_fits = str(fits['neighbourhood'].iloc[0])
REPORT_SLICE = (
    f'Объявлений хотя бы на четырёх гостей {n_fits_four}, '
    f'медиана их цены {median_fits_four:.1f}, '
    f'район первой строки среза — {first_area_fits}.'
)
assert str(n_fits_four) in REPORT_SLICE and first_area_fits in REPORT_SLICE
print(REPORT_SLICE)


## 5. Каких типов комнат больше


In [ ]:
room_counts = df['room_type'].value_counts()
top_room = str(room_counts.idxmax())
n_top_room = int(room_counts.max())
REPORT_ROOMS = f'Чаще всего встречается тип {top_room}: таких объявлений {n_top_room}.'
assert top_room in REPORT_ROOMS and str(n_top_room) in REPORT_ROOMS
print(REPORT_ROOMS)


## 6. Медиана цены у каждого типа комнаты


In [ ]:
median_by_room = df.groupby('room_type')['price'].median()
dearest_room = str(median_by_room.idxmax())
dearest_median = float(median_by_room.max())
REPORT_TYPE = f'Выше всех медиана цены у типа {dearest_room}: {dearest_median:.1f}.'
assert dearest_room in REPORT_TYPE
print(median_by_room)
print(REPORT_TYPE)


## 7. Пять самых дорогих объявлений


In [ ]:
top5 = (
    df.sort_values('price', ascending=False)
    .head(5)[['listing_id', 'neighbourhood', 'room_type', 'price']]
)
row0 = top5.iloc[0]
REPORT_TOP = (
    f'Самое дорогое объявление — {row0.listing_id} в районе {row0.neighbourhood}, '
    f'цена {float(row0.price):.0f}.'
)
assert str(row0.listing_id) in REPORT_TOP
print(top5)
print(REPORT_TOP)


## 8. Середина распределения цены


In [ ]:
q1_price = float(df['price'].quantile(0.25))
q3_price = float(df['price'].quantile(0.75))
mid_width = q3_price - q1_price
REPORT_MIDDLE = (
    f'Нижний квартиль цены {q1_price:.1f}, верхний {q3_price:.1f}, '
    f'середина занимает {mid_width:.1f}.'
)
assert abs(mid_width - (q3_price - q1_price)) < 0.01
print(REPORT_MIDDLE)


## 9. Какой числовой признак ближе к цене


In [ ]:
corr_acc = float(df['accommodates'].corr(df['price']))
corr_bed = float(df['bedrooms'].corr(df['price']))
closer_feature = 'accommodates' if abs(corr_acc) >= abs(corr_bed) else 'bedrooms'
REPORT_CORR = (
    f'Корреляция числа гостей с ценой {corr_acc:.3f}, числа спален с ценой {corr_bed:.3f}. '
    f'Ближе к цене столбец {closer_feature}. '
    'Корреляция на этой таблице не доказывает причину: гости не создают цену только потому, '
    'что числа меняются вместе.'
)
assert closer_feature in REPORT_CORR
print(REPORT_CORR)


## 10. Самый наполненный район


In [ ]:
area_counts = df['neighbourhood'].value_counts()
busy_area = str(area_counts.idxmax())
busy = df.loc[df['neighbourhood'] == busy_area]
n_busy = int(len(busy))
median_busy = float(busy['price'].median())
n_busy_above = int((busy['price'] > price_median).sum())
vs = 'выше' if median_busy > price_median else 'не выше'
REPORT_AREA = (
    f'Больше всего объявлений в районе {busy_area}: их {n_busy}, '
    f'медиана цены {median_busy:.1f}, дороже медианы всей таблицы {n_busy_above} объявлений. '
    f'Медиана этого района {vs} медианы всей таблицы.'
)
assert busy_area in REPORT_AREA and str(n_busy) in REPORT_AREA
print(REPORT_AREA)


## 11. Один и тот же тип комнаты внутри района и вне его


In [ ]:
inside = df.loc[(df['room_type'] == top_room) & (df['neighbourhood'] == busy_area), 'price']
outside = df.loc[(df['room_type'] == top_room) & (df['neighbourhood'] != busy_area), 'price']
median_inside = float(inside.median())
median_outside = float(outside.median())
price_gap = median_inside - median_outside
side = 'дороже' if price_gap > 0 else 'дешевле'
REPORT_GAP = (
    f'Тип {top_room} в районе {busy_area} имеет медиану цены {median_inside:.1f}, '
    f'а тот же тип вне этого района — {median_outside:.1f}. '
    f'Внутри района такой тип {side} на {abs(price_gap):.1f}.'
)
assert top_room in REPORT_GAP and busy_area in REPORT_GAP
print(REPORT_GAP)


## 12. Вывод отчёта


In [ ]:
REPORT_FINAL = (
    f'В таблице {n_rows} объявлений, пустых ячеек нет, средняя цена {price_mean:.1f}, '
    f'а медиана {price_median:.1f}: среднее выше, потому что есть дорогие хвосты. '
    f'Самый наполненный район — {busy_area}: объявлений {n_busy}, медиана цены {median_busy:.1f}. '
    f'Тип {top_room} в этом районе имеет медиану {median_inside:.1f}, '
    f'а вне его — {median_outside:.1f}, разница {price_gap:.1f}. '
    'Ограничение одно: это одна таблица объявлений, корреляция не доказывает причину, '
    'и по ней нельзя обещать ту же цену в другом городе.'
)
assert len(REPORT_FINAL) > 180
assert busy_area in REPORT_FINAL and top_room in REPORT_FINAL
print(REPORT_FINAL)
